# aou_covid — v18.7 re-run

Re-runs the pipeline after the pre-index leakage fix (`223f715`) and adds the
wave x exposure interaction test (`02d`).

**What was wrong.** `01_aou_etl.py` STEP 6 built `num_diagnosis` and
`ehr_length_days` with no index-date restriction, so codes accrued *during* the
COVID admission counted toward the propensity model. `num_diagnosis` is the
strongest matching term (pre-match SMD 0.489), so controls were partly selected
to match a product of the outcome.

**Run the cells in order.** Cell 2 may make cell 7 unnecessary.
Read the output of cell 5 before going further — it is the cell that says whether
the fix actually took effect.


## 0. Configuration


In [ ]:
import os, re, subprocess, textwrap, json, sys
import pandas as pd

COHORT   = 'aou_v7'          # workspace is bound to CDR C2022Q4R13 = cdrv7
BRANCH   = 'review/v18.7-reconcile'
REPO_URL = 'https://github.com/Su-informatics-lab/aou_covid.git'
REPO_DIR = os.path.expanduser('~/aou_covid')

# Two different buckets, and confusing them is the whole reason eTable 12b went
# missing. WORKSPACE_BUCKET is this Verily workspace's own bucket, new and empty.
# The Researcher Workbench 1.0 bucket was migrated to a separate bucket, and that
# is where every legacy output lives.
MIGRATION_BUCKET = 'gs://rw-migration-aou-rw-46c7ae9e'
LEGACY   = f'{MIGRATION_BUCKET}/data/covid_sdoh/{COHORT}'
BUCKET   = os.environ.get('WORKSPACE_BUCKET', '')
BDIR     = f'{BUCKET}/data/covid_sdoh/{COHORT}' if BUCKET else ''

def sh(cmd, check=True):
    print(f'$ {cmd}')
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-8000:])
    if p.returncode and p.stderr: print('STDERR:', p.stderr[-4000:])
    if check and p.returncode: raise RuntimeError(f'failed: {cmd}')
    return p

print('legacy   :', LEGACY)
print('workspace:', BUCKET or '(WORKSPACE_BUCKET unset)')
print('cohort   :', COHORT)


## 1. Get the code

Public repo, so no credentials. If the branch is already checked out this just
fast-forwards it.


In [ ]:
# This workspace sits behind a network perimeter, so github.com may be blocked.
# Find out before assuming, and fall back to the bucket if it is.
if not os.path.isdir(REPO_DIR):
    p = sh(f'git clone {REPO_URL} {REPO_DIR}', check=False)
    if p.returncode:
        raise SystemExit(textwrap.dedent(f'''
            git clone was refused -- almost certainly the perimeter.
            Fallback: upload aou_covid.tar.gz to {LEGACY}/code/ from the
            Workbench bucket browser, then run
              gsutil cp {LEGACY}/code/aou_covid.tar.gz ~/ && tar xzf ~/aou_covid.tar.gz -C ~/
        '''))
os.chdir(REPO_DIR)
sh('git fetch --all --prune', check=False)
sh(f'git checkout {BRANCH}', check=False)
sh('git log --oneline -3', check=False)
os.makedirs(f'results/{COHORT}', exist_ok=True)

src = open('01_aou_etl.py').read()
assert '"survey_ord"' in src, 'survey_ord rename missing from 01_aou_etl.py'
assert 'survey_ord' in open('01b_psm.R').read(), 'survey_ord missing from 01b_psm.R'
print('OK: survey_ord rename is consistent across the ETL and MatchIt')


## 2. Probe the bucket first — this may close eTable 12b/12c without a re-run

`02c` has written its outputs to the workspace bucket since the commit that created
it. The four `wave_stratified_*.csv` were never pulled back into git, which is why
eTable 12b/12c are unverifiable from the repository. If they are still in the bucket,
they can be recovered as-is.

These are products of the **old** matching set. Recovering them closes traceability,
not staleness — they get regenerated in cell 7 either way.


In [ ]:
# Confirmed present in the migration bucket on 2026-09-02 by browsing it:
#   wave_stratified_race.csv, wave_stratified_insurance.csv,
#   wave_stratified_race_attenuation.csv, wave_stratified_income.csv,
#   wave_joint_sdoh_{pre_delta,delta,omicron}_coefficients.csv
sh(f'gsutil ls {LEGACY}/wave_*', check=False)
sh(f'gsutil -m cp {LEGACY}/wave_*.csv results/{COHORT}/', check=False)

import glob
for f in sorted(glob.glob(f'results/{COHORT}/wave_*.csv')):
    print('\n===', f)
    print(pd.read_csv(f).to_string(index=False))

print(textwrap.dedent('''
    CHECK THESE AGAINST eTable 12b / 12c BEFORE GOING FURTHER
      pre-Delta  base 3.00 (2.56-3.51)  joint 2.64 (2.23-3.13)  attenuation 11.5%  1,913 strata
      Delta      base 2.98 (1.79-4.96)  joint 2.17 (1.21-3.89)  attenuation 29.1%    301 strata
      Omicron    base 1.65 (1.34-2.03)  joint 1.42 (1.12-1.79)  attenuation 30.1%  1,105 strata
    Medicaid, eTable 12c
      pre-Delta 1.71 (1.43-2.05) / 1.53 (1.22-1.90)   Delta 2.24 (1.22-4.12) / 1.64 (0.78-3.45)
      Omicron   1.46 (1.15-1.86) / 1.13 (0.84-1.53)
    If they match, eTable 12b/12c stop being UNVERIFIED the moment these files
    reach git. The gap was never the bucket -- it was that nobody pulled them
    back into the repository.
'''))


## 3. Snapshot the OLD matching variables, so the change is measurable

Without a baseline there is no way to tell a working fix from a silent no-op.


In [ ]:
BASE = None
p = sh(f'gsutil cp {LEGACY}/06_matching_variables.csv /tmp/06_matching_OLD.csv', check=False)
if p.returncode == 0:
    BASE = pd.read_csv('/tmp/06_matching_OLD.csv')
    print('old matching variables:', BASE.shape, list(BASE.columns))
    print(BASE[[c for c in ('num_diagnosis','ehr_length_days') if c in BASE]].describe())
else:
    print('No baseline. Cell 5 falls back to the published values:')
    print('  pre-match SMD for num_diagnosis = 0.489; case median num_diagnosis = 114')


## 4. ETL — the fixed STEP 6

Takes a while; it is BigQuery-bound.


In [ ]:
sh(f'python 01_aou_etl.py {COHORT.replace("aou_","")}')


## 5. STOP AND READ — did the index restriction actually take effect?

The single check this whole re-run exists for.

| what | expect |
|---|---|
| `num_diagnosis` median | clearly **lower** than before; unchanged means the CASE WHEN is not biting |
| `num_diagnosis` == 0 | now appears — people with no pre-index diagnosis |
| `ehr_length_days` NaN | now appears; `dropna` drops these people, so the cohort shrinks |
| cohort size | **record the loss** — Figure 1 and the Table 1 N both have to follow it |


In [ ]:
NEW = pd.read_csv(f'results/{COHORT}/06_matching_variables.csv')
coh = pd.read_csv(f'results/{COHORT}/01_covid_cohort.csv')
print('rows', len(NEW), ' unique person_id', NEW.person_id.nunique())
assert len(NEW) == NEW.person_id.nunique(), 'STEP 6 returned more than one row per person'
assert 'survey_ord' in NEW.columns, NEW.columns.tolist()

print('\n--- new ---'); print(NEW[['num_diagnosis','ehr_length_days']].describe())
if BASE is not None and 'num_diagnosis' in BASE:
    o, n = BASE.num_diagnosis.median(), NEW.num_diagnosis.median()
    print(f'\nnum_diagnosis median  old {o:.0f} -> new {n:.0f}   ({100*(n-o)/o:+.1f}%)')
    if n >= o: print('*** WARNING: did not fall. Check the CASE WHEN in match_sql before continuing. ***')

print('\nzeros in num_diagnosis :', int((NEW.num_diagnosis == 0).sum()))
print('NaN in ehr_length_days :', int(NEW.ehr_length_days.isna().sum()))
cc = NEW.dropna(subset=['survey_ord','num_diagnosis','ehr_length_days'])
print(f'complete matching vars : {len(cc):,} / {len(NEW):,}  (lost {len(NEW)-len(cc):,})')
cases = set(coh.loc[coh.severity == 1, 'person_id'])
print(f'cases retained         : {cc.person_id.isin(cases).sum():,}   (published: 4,064)')
print('\n^ if the case count moved, Figure 1 and every N in the paper move with it.')


## 6. Matching


In [ ]:
sh(f'Rscript 01b_psm.R {COHORT}')
smd = pd.read_csv(f'results/{COHORT}/07c_smd_pre_matching.csv')
print(smd.to_string(index=False))
print('\npublished pre-match SMD for num_diagnosis was 0.489; it should now be smaller.')


## 7. Models, sensitivity, tables


In [ ]:
for cmd in [f'Rscript 02_models.R {COHORT}',
            f'Rscript 02b_variance_sensitivity.R {COHORT}',
            f'Rscript 02c_wave_stratified_race_insurance.R {COHORT}',
            f'python 01c_sensitivity_etl.py {COHORT.replace("aou_","")}',
            f'Rscript 03_sensitivity.R {COHORT}',
            f'python 04_tables.py {COHORT}']:
    sh(cmd)


## 8. NEW — wave x exposure interaction (D11)

Closes the third Introduction question, which the Results currently declines
("not compared formally"). One pooled model per exposure; the primary test is a
person_id-clustered Wald test on the interaction block, matching how every other
estimate in the study is clustered.

Power estimated from the published wave-stratified estimates: **race x wave z ~ 4.5**
(pre-Delta vs Omicron), **income x wave z ~ 1.0**. Expect race to be significant and
income not to be — which is what the paper already claims in words.


In [ ]:
sh(f'Rscript 02d_wave_interaction.R {COHORT}')
print(pd.read_csv(f'results/{COHORT}/wave_interaction_tests.csv').to_string(index=False))


## 9. The numbers that decide what has to change in the manuscript


In [ ]:
R = f'results/{COHORT}'
def show(path, **kw):
    try:
        d = pd.read_csv(path)
        for k, v in kw.items(): d = d[d[k].astype(str).str.contains(v, na=False)]
        print(d.to_string(index=False))
    except Exception as e: print(f'  [{path}: {e}]')

print('=== chronic pulmonary / mild liver: still inverse? ===')
print('If these cross toward 1.0, the Discussion paragraph explaining them as a',
      'normal consequence of encounter-density matching must be deleted --',
      'the leakage was the explanation.')
show(f'{R}/base_model_coefficients.csv', variable='Pulmonary|Liver_Disease_Mild')

print('\n=== profile odds-ratio contrast: 1.78 finally gets an interval ===')
show(f'{R}/profile_odds_ratio_contrast.csv')

print('\n=== joint SDoH ===')
show(f'{R}/joint_sdoh_coefficients.csv', variable='f.insurance|f.income|f.employment|f.housing')

print('\n=== race attenuation ===')
show(f'{R}/race_attenuation_table.csv')


## 10. Push everything back, and say what still has to reach git

The bucket is not the repository. That distinction is exactly how eTable 12b/12c
went missing the first time.


In [ ]:
for dest in [d for d in (BDIR, f'{LEGACY}/rerun_v18.7') if d]:
    sh(f'gsutil -m cp {R}/*.csv {dest}/', check=False)
    print('uploaded to', dest)

print(textwrap.dedent('''
    NEXT, OFF-PLATFORM
    1. Pull results/ down and git add it. The four wave_stratified_*.csv are the
       ones that close eTable 12b/12c; the bucket is not the repository.
    2. python 05_figures.py ; python 06_supplement.py ; python make_figures.py
       Do NOT redraw Figure 1 or Figure 2 -- those are the draw.io originals.
    3. bash analysis/gate.sh check   (it will fail widely; that is correct)
    4. bash analysis/gate.sh ledger
    5. Re-run MarketScan on Quartz.
'''))
